# From Polysomnography to Machine Learning
## Automated detection of REM Sleep Behavior Disorder (RBD) — *vibe-coding edition*

**AOC–IFCN Workshop** · Instructor: Po-Yu Lin, MD (National Cheng Kung University Hospital)

*This notebook was written by **Claude Code** (an AI coding assistant); the instructor only
provided the requirements and feedback.*

RBD is one of the strongest early warning signs of Parkinson's disease; its hallmark is
**REM sleep without atonia (RSWA)** — chin-muscle tone that should vanish in REM is
abnormally preserved. You will go from raw sleep signals → a rule-based detector → a
small deep-learning model → model interpretation.

**How this notebook works.** Many code cells are **left blank on purpose**. Above each
blank is a short explanation and an **example prompt** — paste it into Colab's built-in
AI (*Gemini*) to generate the code, run it, and fix errors by pasting them back. This
is "vibe coding": you steer, the AI types. (A fully worked *answers* notebook exists if
you get stuck.)

*Full references are at the end.*

---
## Warm-up · How to vibe-code in Colab

A few habits make AI-assisted coding work well:

1. **Generate in place.** Click the **✨ / "Generate"** button in a code cell, type what
   you want, and let Gemini write it. Run with `Shift+Enter`.
2. **Paste errors back.** If a cell turns red, **copy the whole error message** and paste
   it to Gemini with "fix this". Errors are information, not failure.
3. **Be specific.** Say the inputs, the shapes, the library, and what the output should
   look like. Vague prompts give vague code.
4. **Iterate.** "Now make the font bigger", "use a different colormap", "add a title" —
   refine in small steps.

**Try it now.** The prompt below draws a confusion matrix from four numbers. Generate it,
run it, then ask Gemini to **change the colour** (e.g. to `Greens`) and re-run.

> *Prompt:* "Draw a simple 2×2 confusion matrix for a test that sorts people into
> 'control' versus 'RBD'. Use these four counts: 18 controls correctly called control, 2
> controls wrongly called RBD, 1 patient missed (called control), and 9 patients correctly
> called RBD. Show it as a small coloured grid with each number written inside its square,
> the columns labelled by what the test predicted and the rows by the truth, and give it a
> title."

In [ ]:
# 🧑‍💻 Your turn — generate this cell with Colab's Gemini using the prompt above.
# (Click "Generate" / the ✨ in the cell, paste the prompt, run it; if it errors,
#  paste the red error text back to Gemini and ask it to fix it.)

---
## Module 0 · Setup (run this first — do not edit)

Runs in **Google Colab** or **locally**. `RUNTIME` auto-detects; on Colab the packages
install automatically. These setup + download + file-reading cells are **given** — just
run them.

### Step 1 — where are you running?

In [ ]:
RUNTIME = "auto"  # "auto" | "colab" | "local"
DATA_DIR = None  # None = default folder for your runtime; or set your own path string

import os, sys, subprocess  # standard-library helpers
try:
    import google.colab  # only importable on Colab
    on_colab = True  # we are on Colab
except ImportError:
    on_colab = False  # not on Colab
if RUNTIME == "auto":  # resolve "auto"
    RUNTIME = "colab" if on_colab else "local"  # detect runtime

if DATA_DIR is None:  # default data folder
    DATA_DIR = "/content/workshop_data" if RUNTIME == "colab" else "workshop_data"  # Colab vs local
DATA = os.path.abspath(DATA_DIR)  # absolute path used everywhere below
os.makedirs(DATA, exist_ok=True)  # create it if needed

if RUNTIME == "colab":  # Colab lacks mne/h5py -> install quietly
    print("Colab detected -- installing mne and h5py ...")  # progress
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mne", "h5py"],  # install
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)  # hide pip noise
else:  # local -> use your prepared environment
    print("Local run -- using your current Python environment (see setup_instructions.md)")  # reminder
print(f"runtime = {RUNTIME}  |  data dir = {DATA}")  # confirm

### Step 2 — import libraries

In [ ]:
import urllib.request  # download over HTTPS
import numpy as np  # arrays and math
import matplotlib.pyplot as plt  # plotting
import h5py  # read the HDF5 dataset
import mne  # PSG / EDF handling + filtering
mne.set_log_level("WARNING")  # quieten MNE
import torch  # deep-learning library (Modules 2+)
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available())  # show versions/GPU

### Step 3 — download the workshop data (given)

In [ ]:
BASE = "https://github.com/Po-yuLin/AOC-IFCN_workshop/releases/download/data-v1"  # release URL
FILES = [  # published assets:
    "workshop_data_fp16.h5",  # dataset: 3-second REM mini-epochs + labels
    "clip_control.edf", "clip_control.txt",  # raw PSG clip from a control (n1)
    "clip_rbd.edf", "clip_rbd.txt",  # raw PSG clip from an RBD patient (rbd2)
]
for fname in FILES:  # each file
    dst = os.path.join(DATA, fname)  # local path
    if not os.path.exists(dst):  # skip if already present
        print("downloading", fname, "..."); urllib.request.urlretrieve(f"{BASE}/{fname}", dst)  # fetch
print("Files ready:", sorted(os.listdir(DATA)))  # list

---
# Module 1 · From PSG signals to a rule-based RSWA detector

For RBD, three channels matter: **EEG** (sleep stage — we only care about REM), **EOG**
(eye movements that define REM), and the **chin EMG** (should be silent in REM;
activity there = RSWA). We will read one recording, find REM, turn the chin EMG into an
activity measure, score RSWA per **3-second mini-epoch**, and build a rule.

### 1.1 — Open a clip and find the channels
First we open a short raw clip (given). Then **you** identify which of its channels are
the EEG, EOG and chin EMG. This clip carries just those three; full clinical montages
have many more (and name them inconsistently), so picking channels by name is a useful
habit.

In [ ]:
raw = mne.io.read_raw_edf(os.path.join(DATA, "clip_control.edf"), preload=True, verbose=False)  # open clip
print("Channels:", raw.ch_names)  # what's in this recording
print("Sampling rate:", raw.info["sfreq"], "Hz | duration:", round(raw.times[-1], 1), "s")  # rate + length

Different recordings don't label their channels the same way. Before we pick channels
automatically, let's see the problem: list the **chin-EMG channel name** stored for every
subject in the full dataset — they are *not* all the same (one recording even calls it
`milo`). This is exactly why we cannot hard-code a single name.

In [ ]:
with h5py.File(os.path.join(DATA, "workshop_data_fp16.h5"), "r") as h:  # open the dataset read-only
    for grp in ("control", "rbd"):  # look in both groups
        for subj in h[grp]:  # each subject id in the group
            print(f"{subj:8s} chin-EMG channel = {h[grp][subj].attrs['emg_ch']}")  # print the stored name

So instead of assuming a name, we keep a short **list of the names each channel might go
by** and pick whichever one this particular recording actually has. This channel lookup is
**given** — the names it finds feed everything downstream — so just run it.

In [ ]:
CH = {"eeg": ["C4-A1", "C4-P4", "Fp2-F4", "F4-C4"],  # names the EEG might go by
      "eog": ["ROC-LOC", "EOG dx", "EOG-L", "EOG sin"],  # names the EOG might go by
      "emg": ["EMG1-EMG2", "EMG-EMG", "EMG", "CHIN-0", "CHIN1", "milo"]}  # names the chin EMG might go by
eeg_ch = next(c for c in raw.ch_names if c in CH["eeg"])  # the EEG channel in this clip
eog_ch = next(c for c in raw.ch_names if c in CH["eog"])  # the EOG channel in this clip
emg_ch = next(c for c in raw.ch_names if c in CH["emg"])  # the chin-EMG channel in this clip
print("EEG:", eeg_ch, "| EOG:", eog_ch, "| chin EMG:", emg_ch)  # what we found

### 1.2 — Find the sleep stages, and look at REM vs non-REM
The sleep stage of every 30-second epoch is listed in the recording's **`.txt` file**
(lines contain a `SLEEP-…` token and an `HH:MM:SS` clock time). We parse it and draw the
**hypnogram** (stage over time). This staging read is **given** — it produces the REM and
non-REM epoch lists the rest of Module 1 relies on — so just run it and look at the night.

In [ ]:
mdate = raw.info["meas_date"]  # recording start (datetime)
rec0 = mdate.hour*3600 + mdate.minute*60 + mdate.second  # start seconds-since-midnight
stg = []  # (relative_seconds, stage)
for line in open(os.path.join(DATA, "clip_control.txt"), errors="ignore"):  # read the .txt
    p = line.split()  # split on whitespace
    if len(p) < 5:  # skip header/short lines
        continue  # next line
    stage = next((x for x in p if x.startswith("SLEEP-")), None)  # stage token
    clock = next((x for x in p if ":" in x and len(x) == 8), None)  # HH:MM:SS token
    if stage and clock:  # keep lines with both
        h, m, s = clock.split(":")  # split the clock
        rel = int(h)*3600 + int(m)*60 + int(s) - rec0  # seconds after recording start
        if rel < -3600:  # midnight wrap
            rel += 86400  # add 24 h
        stg.append((rel, stage))  # store
rem_starts = [t for t, s in stg if s == "SLEEP-REM" and t >= 0]  # REM epoch starts
nonrem_starts = [t for t, s in stg if s.startswith("SLEEP-S") and t >= 0]  # non-REM epoch starts
print("REM epochs:", len(rem_starts), "| non-REM epochs:", len(nonrem_starts))  # counts

LEVELS = {"SLEEP-REM": 4, "SLEEP-S1": 3, "SLEEP-S2": 2, "SLEEP-S3": 1, "SLEEP-S4": 0}  # y per stage
xs = [t/60 for t, s in stg if s in LEVELS]; ys = [LEVELS[s] for t, s in stg if s in LEVELS]  # points
plt.figure(figsize=(10, 2.6)); plt.step(xs, ys, where="post", lw=1.2)  # staircase
plt.yticks(list(LEVELS.values()), list(LEVELS.keys())); plt.xlabel("minutes")  # labels
plt.title("Hypnogram of the clip"); plt.tight_layout(); plt.show()  # render

**Given helper — filtering.** This band-passes one channel and resamples it to a common
200 Hz. Just run it — the next cell is the first that uses it.

In [ ]:
FS = 200  # common sampling rate (Hz)
def prep_channel(raw, ch, lo, hi):  # band-pass one channel and resample to FS
    sf0 = raw.info["sfreq"]  # native rate
    x = raw.get_data(picks=[ch])[0] * 1e6  # channel in microvolts
    hi = min(hi, sf0/2 - 1)  # stay below Nyquist
    x = mne.filter.filter_data(x, sf0, lo, hi, verbose=False)  # band-pass at native rate
    if 50.0 < sf0/2:  # if 50 Hz is below Nyquist,
        x = mne.filter.notch_filter(x, sf0, [50.0], verbose=False)  # remove mains hum
    return mne.filter.resample(x, up=FS, down=int(round(sf0)), verbose=False)  # resample -> 200 Hz

Now compare **one REM epoch** with **one non-REM epoch** on three channels (EEG, EOG,
chin EMG). In REM you should see eye movements (EOG) and a **quiet chin EMG**; in
non-REM the EMG is usually a bit higher and steadier.

> *Prompt:* "Using the filtering helper we defined above, prepare the three channels for
> viewing: the brain wave (keep roughly 0.3–70 Hz), the eye movement (0.3–35 Hz) and the
> chin muscle (10–100 Hz). Take the first complete 30-second REM epoch and the first
> complete non-REM epoch. Lay out a grid three rows tall (brain wave, eye movement, chin
> muscle) and two columns wide (non-REM on the left, REM on the right), each panel showing
> its 30 seconds on a matching vertical scale — so we can see that REM has eye movements
> but a quiet chin."

In [ ]:
# 🧑‍💻 Your turn — generate this cell with Colab's Gemini using the prompt above.
# (Click "Generate" / the ✨ in the cell, paste the prompt, run it; if it errors,
#  paste the red error text back to Gemini and ask it to fix it.)

### 1.3 — Turn the chin EMG into an "activity" amplitude
Following the Ferri REM Atonia Index method, the chin EMG becomes a muscle-activity
measure in four steps: **(1)** the filtered signal → **(2)** rectify (absolute value) →
**(3)** average into 0.1-second bins → **(4)** subtract a 60-second rolling minimum
(the background/noise floor). This step is **given** — it produces the background-corrected
activity signal the next cell scores — so just run it and watch the four panels transform
the signal.

In [ ]:
from scipy.ndimage import minimum_filter1d  # rolling-minimum filter
emg = prep_channel(raw, emg_ch, 1.0, 100.0)  # filtered chin EMG at 200 Hz (uV)
rect = np.abs(emg)  # (2) rectified
bin01 = FS // 10  # samples per 0.1-s bin (=20)
n01 = len(rect) // bin01  # number of whole bins
a01 = rect[:n01*bin01].reshape(n01, bin01).mean(1)  # (3) 0.1-s mean amplitude
baseline = minimum_filter1d(a01, size=600)  # 60-s rolling minimum (=600 bins)
a01c = np.clip(a01 - baseline, 0, None)  # (4) background-corrected activity
w0 = rem_starts[0]; w1 = min(w0 + 60, len(a01)//10)  # a 60-second window from the first REM epoch
fig, ax = plt.subplots(4, 1, figsize=(10, 8), sharex=True)  # four stacked panels
te = np.arange(w0*FS, w1*FS) / FS  # time axis for the 200-Hz signals
ax[0].plot(te, emg[w0*FS:w1*FS], lw=0.5); ax[0].set_ylabel("(1) filtered")  # raw filtered
ax[1].plot(te, rect[w0*FS:w1*FS], lw=0.5); ax[1].set_ylabel("(2) rectified")  # rectified
tb = np.arange(w0*10, w1*10) / 10  # time axis for the 0.1-s series
ax[2].plot(tb, a01[w0*10:w1*10], lw=0.8); ax[2].set_ylabel("(3) 0.1-s mean")  # binned
ax[3].plot(tb, a01c[w0*10:w1*10], lw=0.8); ax[3].set_ylabel("(4) minus 60-s min")  # corrected
ax[3].set_xlabel("seconds"); ax[0].set_title("chin EMG -> activity amplitude"); plt.tight_layout(); plt.show()

### 1.4 — Score each 3-second mini-epoch, and see which are phasic vs tonic
Each REM 30-s epoch is split into ten **3-second mini-epochs**. On the
background-corrected amplitude we mark a mini as **phasic** (a burst > 3 µV), **tonic**
(> 2 µV for ≥ 50 % of the mini), or **"any"** (phasic OR tonic). Numbers alone are
abstract — **colour the timeline** so you can see which minis are phasic vs tonic.

*(Guideline: phasic ≈ 0.1–14.9 s bursts > 4× background; tonic > 2× background for >50 %
of a 30-s epoch; SINBAR chin "any" cutoff ≈ 18 %. We use calibrated absolute-µV
thresholds — see the answers notebook / references.)*

> *Prompt:* "Score REM in three-second mini-epochs using the background-corrected activity
> signal from the previous step. Call a mini **phasic** if it contains a brief tall spike
> (its peak rises above 3 microvolts), and **tonic** if the muscle stays raised — above 2
> microvolts for at least half of the mini. 'Any activity' means phasic or tonic. Then draw
> the activity signal over the first minute-and-a-half of REM and shade each three-second
> mini: red where it's phasic, orange where it's tonic only, pale grey where it's neither.
> Add a legend, and print what percentage of the REM minis showed any activity."

In [ ]:
# 🧑‍💻 Your turn — generate this cell with Colab's Gemini using the prompt above.
# (Click "Generate" / the ✨ in the cell, paste the prompt, run it; if it errors,
#  paste the red error text back to Gemini and ask it to fix it.)

### 1.5 — The whole dataset and the rule-based diagnosis
The full dataset stores every subject's REM minis plus a whole-night **"any" density**.
We read those (given), then **you** build the classic rule: **density > cutoff →
diagnose RBD**, and check how well one number separates 8 controls from 21 patients.

In [ ]:
any_density, truth, names = [], [], []  # density %, true label (1=RBD), subject id
with h5py.File(os.path.join(DATA, "workshop_data_fp16.h5"), "r") as h:  # open dataset (reading = given)
    for grp, is_rbd in (("control", 0), ("rbd", 1)):  # controls then patients
        for subj in h[grp]:  # each subject
            any_density.append(float(h[grp][subj].attrs["any_density"]))  # stored "any" density
            truth.append(is_rbd); names.append(subj)  # label + id
any_density = np.array(any_density); truth = np.array(truth)  # to arrays
print("controls:", any_density[truth == 0].round(1)); print("RBD     :", any_density[truth == 1].round(1))  # peek

For every subject we now have one number: the percentage of their REM sleep that showed any
muscle activity. The plot below is **given** — it shows those as dots, controls on the left
and RBD on the right, with the 18 % clinical cut-off drawn in. (You'll build a richer
version of this same picture yourself in Module 2.)

In [ ]:
plt.figure(figsize=(5.5, 4))  # new figure
for gi, (lbl, mask, col) in enumerate([("control", truth == 0, "steelblue"), ("RBD", truth == 1, "crimson")]):  # groups
    ys = any_density[mask]; xs = gi + np.linspace(-0.12, 0.12, len(ys))  # jittered x
    plt.scatter(xs, ys, c=col, label=lbl, zorder=3)  # points
plt.axhline(18, ls="--", c="gray", label="cutoff = 18%")  # cutoff
plt.xticks([0, 1], ["control", "RBD"]); plt.ylabel("'any' density (%)")  # labels
plt.legend(); plt.title("Rule-based separation"); plt.tight_layout(); plt.show()  # render

> *Prompt:* "Turn that picture into a diagnosis: call anyone above 18 percent 'RBD' and
> everyone else a 'control'. Compare against the true diagnoses and report two numbers — the
> fraction of real patients the rule catches (sensitivity) and the fraction of real controls
> it correctly clears (specificity). Then name the mistakes: which patients were missed, and
> which controls were false alarms."

In [ ]:
# 🧑‍💻 Your turn — generate this cell with Colab's Gemini using the prompt above.
# (Click "Generate" / the ✨ in the cell, paste the prompt, run it; if it errors,
#  paste the red error text back to Gemini and ask it to fix it.)

There is **no single perfect cutoff** — lower catches more patients but raises false
alarms. Draw the whole trade-off as an **ROC curve** (scikit-learn) and mark a few cutoffs.

> *Prompt:* "There's no single perfect cut-off — a lower one catches more patients but
> raises more false alarms. Show that whole trade-off as an ROC curve: sweep the threshold
> and plot sensitivity against the false-alarm rate (one minus specificity). Put the area
> under the curve in the legend, draw the diagonal 'no better than chance' line, and mark
> where a few specific cut-offs land — 6, 14, 18 and 30 percent."

In [ ]:
# 🧑‍💻 Your turn — generate this cell with Colab's Gemini using the prompt above.
# (Click "Generate" / the ✨ in the cell, paste the prompt, run it; if it errors,
#  paste the red error text back to Gemini and ask it to fix it.)

---
# Module 2 · Teaching a 1D-CNN to detect RSWA

Each mini-epoch is a `(3 channels, 600 samples)` image (EEG, EOG, chin EMG over 3 s at
200 Hz); the CNN predicts **"any" RSWA — yes/no**, the same per-mini label the rule uses.
We use a small **multi-scale (Inception-style)** network — parallel kernels of several
widths catch both fast EMG bursts and slow EEG rhythms. Load the data (given):

In [ ]:
H5 = os.path.join(DATA, "workshop_data_fp16.h5")  # dataset path
X_list, y_list, subj_list = [], [], []  # signals, labels, subject-id per mini
with h5py.File(H5, "r") as h:  # open (reading = given)
    for grp in ("control", "rbd"):  # both groups
        for subj in h[grp]:  # each subject
            sig = h[grp][subj]["signals"][:].astype("float32")  # (n,3,600) fp16 -> float32 (physical uV)
            lab = h[grp][subj]["labels"][:].astype("int64")  # (n,) "any" label
            X_list.append(sig); y_list.append(lab); subj_list += [subj] * len(lab)  # collect
X = np.concatenate(X_list); y = np.concatenate(y_list); subj_arr = np.array(subj_list)  # stack
print("total minis:", X.shape, "| positive fraction:", round(float(y.mean()), 3))  # summary

### 2.1 — Split by subject, then look at where the splits fall
We split **by subject** (never mix a subject across splits). Two design choices matter:

- **Training must see hard negatives, not just easy positives.** If the model only ever
  saw quiet controls and floridly abnormal RBD nights, it learns "any burst → RBD" and
  then **over-calls** borderline subjects. So training keeps the low-RSWA RBD nights (the
  near-silent `rbd17`, plus `rbd10`, `rbd4`) as *negative*-rich examples, alongside clear
  controls and the two awkward controls the model tends to over-call — the very elevated
  `n4` and `n10`.
- **Test is 4 controls + 5 RBD, and stresses the boundary.** Its controls are `n3`, `n5`,
  `n8` and the elevated `n11` (the one the rule may false-alarm on); its RBD side pairs the
  low-RSWA nights the rule may miss (`rbd6`, `rbd15`) with clear positives (`rbd5`, `rbd9`,
  `rbd21`). The genuinely *awkward* cases stay in **training**, not test: `n4` and `n10`
  (controls the model over-calls) and `rbd17` (an RBD that looks like a healthy sleeper) —
  putting them in the test would only add noise to the score.

The split, an `EPOCHS` knob (lower it for a quicker run) and a tiny `SMALL_N` "small-data"
subset (used next to provoke overfitting) are given, along with the global normalization:

In [ ]:
TRAIN_SUBJ = ["n1", "n4", "n10",  # 3 controls (n4 = very elevated; n10 = the control the model tends to over-call) +
              "rbd1", "rbd2", "rbd3", "rbd4", "rbd7", "rbd8", "rbd10", "rbd12",  # 15 RBD across the whole
              "rbd13", "rbd14", "rbd16", "rbd17", "rbd19", "rbd20", "rbd22"]  # RSWA range (incl. near-silent rbd17)
VAL_SUBJ = ["n2", "rbd18"]  # watched during training (we checkpoint on best val loss)
train_idx = np.where(np.isin(subj_arr, TRAIN_SUBJ))[0]  # training minis
val_idx = np.where(np.isin(subj_arr, VAL_SUBJ))[0]  # validation minis
test_idx = np.where(~np.isin(subj_arr, TRAIN_SUBJ + VAL_SUBJ))[0]  # rest = held-out test: n3,n5,n8,n11 + rbd5,6,9,15,21

mu = X[train_idx].mean(axis=(0, 2), keepdims=True)  # per-channel mean over TRAIN minis (one global scale)
sd = X[train_idx].std(axis=(0, 2), keepdims=True) + 1e-6  # per-channel std over TRAIN minis
X = (X - mu) / sd  # standardise all subjects with the same scale (keeps cross-subject amplitude)

EPOCHS = 20  # training length -- 20 is plenty here (later epochs barely change); lower for speed
rng = np.random.default_rng(0)  # fixed seed so the small-data demo is reproducible
SMALL_N = 10  # tiny "small data" subset -- only ~10 minis, so the model overfits hard and fast
small_idx = rng.choice(train_idx, size=min(SMALL_N, len(train_idx)), replace=False)  # a tiny slice of train minis
print("test subjects :", sorted(set(subj_arr[test_idx])))  # who is held out
print("small subset  :", len(small_idx), "minis sampled from the training pool")  # the overfit set
print("minis train/val/test:", len(train_idx), len(val_idx), len(test_idx))  # sizes

> *Prompt:* "Redraw the same control-versus-RBD dot plot from Module 1 (two columns, the
> dashed line at 18 percent). This time, put a big soft-coloured halo behind each subject's
> dot to show which group it belongs to: green for the subjects we train on, orange for the
> few we watch during training, and red for the held-out test subjects we only judge at the
> very end. Add a legend for the three colours."

In [ ]:
# 🧑‍💻 Your turn — generate this cell with Colab's Gemini using the prompt above.
# (Click "Generate" / the ✨ in the cell, paste the prompt, run it; if it errors,
#  paste the red error text back to Gemini and ask it to fix it.)

### 2.2 — The model and the training loop (given)
Three multi-scale blocks → global average pool → linear head. The training loop keeps
the **best-validation checkpoint** and returns `(best_model, last_model, history)`; pass
any architecture via `model_fn`. Just run these two cells.

In [ ]:
import torch.nn as nn  # network layers
import torch.nn.functional as F  # functional ops
torch.manual_seed(0)  # reproducible weights

class InceptionBlock(nn.Module):  # parallel convolutions of several widths
    def __init__(self, c_in, c_out):  # c_out maps per branch
        super().__init__()  # init parent
        self.short = nn.Conv1d(c_in, c_out, 3, padding=1)  # narrow -> fast patterns
        self.mid = nn.Conv1d(c_in, c_out, 9, padding=4)  # medium
        self.long = nn.Conv1d(c_in, c_out, 19, padding=9)  # wide -> slow patterns
        self.pool = nn.MaxPool1d(3, stride=1, padding=1)  # pooling branch
        self.poolproj = nn.Conv1d(c_in, c_out, 1)  # 1x1 after pooling
    def forward(self, x):  # (batch, c_in, time)
        return F.relu(torch.cat([self.short(x), self.mid(x), self.long(x), self.poolproj(self.pool(x))], 1))  # concat

class InceptionCNN(nn.Module):  # small multi-scale network
    def __init__(self, n_ch=3, n_classes=2, c=16):  # channels/classes/width
        super().__init__()  # init parent
        self.stem = nn.Conv1d(n_ch, c, 7, stride=2, padding=3)  # downsample by 2
        self.inc1 = InceptionBlock(c, c); self.inc2 = InceptionBlock(4*c, c); self.inc3 = InceptionBlock(4*c, c)  # blocks
        self.pool = nn.MaxPool1d(2); self.gap = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(4*c, n_classes)  # head
    def forward(self, x, return_feat=False):  # (batch,3,600)
        x = F.relu(self.stem(x)); x = self.pool(self.inc1(x)); x = self.pool(self.inc2(x))  # stem + 2 blocks
        feat = self.inc3(x)  # final feature maps
        logits = self.head(self.gap(feat).squeeze(-1))  # pooled -> class scores
        return (logits, feat) if return_feat else logits  # feats for Grad-CAM

device = "cuda" if torch.cuda.is_available() else "cpu"  # GPU if available
print("training on:", device, "| params:", sum(p.numel() for p in InceptionCNN().parameters()))  # info

In [ ]:
from tqdm.auto import tqdm  # progress bars
import copy  # to snapshot best weights

def make_loader(idx, batch=64, shuffle=True, Xsrc=None):  # DataLoader over mini indices
    src = X if Xsrc is None else Xsrc  # which signals (lets us ablate channels later)
    ds = torch.utils.data.TensorDataset(torch.tensor(src[idx]), torch.tensor(y[idx]))  # pair
    return torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=shuffle)  # batches

def evaluate(model, idx, Xsrc=None):  # mean loss + accuracy, batched
    model.eval(); tot = correct = n = 0  # totals
    with torch.no_grad():  # no gradients
        for xb, yb in make_loader(idx, 256, False, Xsrc):  # batches
            xb, yb = xb.to(device), yb.to(device); lo = model(xb)  # forward
            tot += F.cross_entropy(lo, yb, reduction="sum").item(); correct += (lo.argmax(1) == yb).sum().item(); n += len(yb)  # accumulate
    return tot/n, correct/n  # mean loss, accuracy

def train_model(train_idx, val_idx, model_fn=InceptionCNN, epochs=EPOCHS, lr=1e-3, augment=False, Xsrc=None):  # train + log
    model = model_fn().to(device); opt = torch.optim.Adam(model.parameters(), lr=lr)  # fresh model + optimiser
    loader = make_loader(train_idx, Xsrc=Xsrc); hist = {"tr_loss": [], "va_loss": [], "tr_acc": [], "va_acc": []}  # setup
    best_vl, best_ep, best_state = 1e9, 0, copy.deepcopy(model.state_dict())  # best-val checkpoint
    for ep in range(epochs):  # epochs
        model.train()  # train mode
        for xb, yb in tqdm(loader, desc=f"epoch {ep+1}/{epochs}", leave=False):  # batches (live bar)
            xb, yb = xb.to(device), yb.to(device)  # to device
            if augment: xb = augment_batch(xb)  # optional augmentation
            opt.zero_grad(); loss = F.cross_entropy(model(xb), yb); loss.backward(); opt.step()  # one step
        tl, ta = evaluate(model, train_idx, Xsrc); vl, va = evaluate(model, val_idx, Xsrc)  # epoch metrics
        for k, v in zip(hist, (tl, vl, ta, va)): hist[k].append(v)  # record
        print(f"  epoch {ep+1:2d}/{epochs}  train loss {tl:.3f} (acc {ta:.2f})   val loss {vl:.3f} (acc {va:.2f})")  # per-epoch log
        if vl < best_vl: best_vl, best_ep, best_state = vl, ep, copy.deepcopy(model.state_dict())  # keep best
    best = model_fn().to(device); best.load_state_dict(best_state)  # rebuild best
    print(f"best epoch {best_ep+1} (val acc {hist['va_acc'][best_ep]:.2f}) | last (val acc {hist['va_acc'][-1]:.2f})")  # report
    return best, model, hist  # best, last, curves

### 2.3 — A helper to score whole subjects on the test set (given)
This turns a model's per-mini predictions into a **per-subject "any" density**, applies
the 18 % cutoff, and — in one call, `report_test(model, "name")` — prints **sensitivity,
specificity and accuracy** for the model **and** the Module-1 rule, and draws their two
confusion matrices side by side. We call it after **every** training below (the overfit
one, the augmented one, the full one), so run this cell now.

In [ ]:
from sklearn.metrics import confusion_matrix  # confusion matrix

def subject_density(model, Xsrc):  # model "any" density (%) per TEST subject
    model.eval(); ps = []  # probabilities
    with torch.no_grad():  # no gradients
        for a in range(0, len(test_idx), 256):  # chunks of test minis
            chunk = test_idx[a:a+256]  # indices
            ps.append(torch.softmax(model(torch.tensor(Xsrc[chunk]).to(device)), 1)[:, 1].cpu().numpy())  # P(any)
    prob = np.concatenate(ps); subs = sorted(set(subj_arr[test_idx]))  # per-mini probs + subjects
    dens = np.array([100 * prob[subj_arr[test_idx] == s].mean() for s in subs])  # density per subject
    return subs, dens  # subjects + densities

def subject_metrics(subs, dens, cut=18):  # confusion matrix + sensitivity/specificity/PPV
    tr = np.array([s.startswith("rbd") for s in subs]); pr = dens > cut  # truth + prediction
    cm = confusion_matrix(tr, pr, labels=[False, True]); tn, fp, fn, tp = cm.ravel()  # 2x2
    sens = tp/(tp+fn) if tp+fn else float("nan"); spec = tn/(tn+fp) if tn+fp else float("nan"); ppv = tp/(tp+fp) if tp+fp else float("nan")  # metrics
    return cm, sens, spec, ppv  # results

def show_cms(cm_model, cm_rule, title_model="CNN"):  # two confusion matrices, side by side, different colours
    fig, ax = plt.subplots(1, 2, figsize=(6.8, 3.2))  # two panels
    for a, (cm, ti, cmap) in zip(ax, [(cm_model, title_model, "Blues"), (cm_rule, "Rule (Module 1)", "Oranges")]):  # model | rule
        a.imshow(cm, cmap=cmap)  # colour counts
        for (r, c), v in np.ndenumerate(cm): a.text(c, r, int(v), ha="center", va="center", fontsize=12)  # numbers
        a.set_xticks([0, 1]); a.set_xticklabels(["ctrl", "RBD"]); a.set_yticks([0, 1]); a.set_yticklabels(["ctrl", "RBD"])  # ticks
        a.set_xlabel("predicted"); a.set_ylabel("true"); a.set_title(ti)  # labels
    plt.tight_layout(); plt.show()  # render

def report_test(model, title="CNN", Xsrc=None):  # test-set confusion matrices + metrics, model vs the rule
    subs, dens = subject_density(model, X if Xsrc is None else Xsrc)  # model "any" density on the held-out subjects
    cm_m, sens_m, spec_m, ppv_m = subject_metrics(subs, dens)  # model confusion matrix + metrics
    rule_d = np.array([any_density[names.index(s)] for s in subs])  # the rule's density, same subjects
    cm_r, sens_r, spec_r, ppv_r = subject_metrics(subs, rule_d)  # rule confusion matrix + metrics
    acc_m = (cm_m[0, 0] + cm_m[1, 1]) / cm_m.sum(); acc_r = (cm_r[0, 0] + cm_r[1, 1]) / cm_r.sum()  # accuracies
    print(f"{title:22s} sensitivity {sens_m:.0%} | specificity {spec_m:.0%} | accuracy {acc_m:.0%}")  # model line
    print(f"{'rule (Module 1)':22s} sensitivity {sens_r:.0%} | specificity {spec_r:.0%} | accuracy {acc_r:.0%}")  # rule line
    show_cms(cm_m, cm_r, title_model=title)  # two confusion matrices side by side
    return dict(subs=subs, dens=dens, sens=sens_m, spec=spec_m, acc=acc_m)  # hand back for reuse

### 2.4 — Watch it overfit (too little data)
Here is your first look at the training loop, **given as a worked example** so you can see
how it is called and how to read the curves. We train on the tiny handful of data and just
watch the **learning curves**: the **training** accuracy climbs toward perfect while the
**validation** accuracy stalls (and the validation *loss* often creeps back up) — that gap is
overfitting, the model memorising instead of learning. (We don't score it on the test set:
our test group is deliberately a bit easy, so even an over-fitted model can look okay there —
the curves are the honest tell.) In the next cells **you** drive the same loop yourself, with
a twist each time.

In [ ]:
model_small, _, hist_small = train_model(small_idx, val_idx, epochs=EPOCHS)  # train on the tiny subset
fig, ax = plt.subplots(1, 2, figsize=(11, 4))  # its own loss + accuracy curves
ax[0].plot(hist_small["tr_loss"], label="train"); ax[0].plot(hist_small["va_loss"], label="val"); ax[0].set_title("loss"); ax[0].legend()  # loss
ax[1].plot(hist_small["tr_acc"], label="train"); ax[1].plot(hist_small["va_acc"], label="val"); ax[1].set_title("accuracy"); ax[1].legend()  # accuracy
plt.suptitle("Small data -> overfitting"); plt.tight_layout(); plt.show()  # render

---
# Module 3 · Fighting overfitting: augmentation, then more data

### 3.1 — Fix the augmentation
Augmentation slightly distorts each training batch so the model can't just memorise it. The
starter below does **three** things — adds background noise, **rescales the amplitude**, and
shifts in time. But **RSWA is judged by amplitude**, so rescaling it would corrupt the very
signal we're trying to detect. **Your job: delete the two amplitude-scaling lines**, leaving
only the noise and the time-shift. (Keep the name `augment_batch` — the training loop calls
it by that name. If you like, ask Gemini to "remove the amplitude scaling from this function".)

In [ ]:
def augment_batch(xb):  # xb: (batch, 3, 600) on the device
    out = xb + 0.1 * torch.randn_like(xb)  # add small Gaussian noise
    gain = 0.9 + 0.2 * torch.rand(out.size(0), 1, 1, device=out.device)  # <-- DELETE: random amplitude scaling
    out = out * gain  # <-- DELETE: rescaling amplitude corrupts the RSWA signal we detect
    shift = int(torch.randint(-20, 21, (1,)).item())  # random time shift (+-0.1 s)
    return torch.roll(out, shifts=shift, dims=-1)  # roll along time

### 3.2 — Small data + augmentation
Train on the same tiny subset again, but with augmentation switched on. Look at its own
learning curves — the overfitting gap should be a little smaller than before.

> *Prompt:* "Train again on the same tiny handful of data, but this time switch on the
> augmentation so every batch is slightly distorted as it trains. Then draw its learning
> curves — the error and the accuracy over the epochs, for both the training and the
> validation group."

In [ ]:
# 🧑‍💻 Your turn — generate this cell with Colab's Gemini using the prompt above.
# (Click "Generate" / the ✨ in the cell, paste the prompt, run it; if it errors,
#  paste the red error text back to Gemini and ask it to fix it.)

### 3.3 — Train on the whole training set
More (and more varied) data is usually the biggest win.

> *Prompt:* "Now train on the whole training set (still with augmentation) and keep the
> result under the name `model_full` — the scoring and interpretation cells below use that
> name. Draw its learning curves — the error and the accuracy over the epochs, for both the
> training and the validation group. With this much data the two curves should track each
> other far more closely than the tiny-data run did."

In [ ]:
# 🧑‍💻 Your turn — generate this cell with Colab's Gemini using the prompt above.
# (Click "Generate" / the ✨ in the cell, paste the prompt, run it; if it errors,
#  paste the red error text back to Gemini and ask it to fix it.)

### 3.4 — Evaluate the full model on the held-out test set (vs the rule)
Now the payoff — this cell is **given, just run it**: we aggregate the full model's per-mini
predictions to a per-subject density, apply the 18 % cutoff, and compare with the Module 1
rule on the **same** test subjects — **two confusion matrices side by side** (CNN in blue,
rule in orange), using the helpers from **2.3**. We keep the returned numbers to reuse in
Module 4.

In [ ]:
full_res = report_test(model_full, "CNN (all data)")  # test confusion matrices + sensitivity/specificity/accuracy vs the rule

---
# Module 4 · Opening the black box

Does the model look where a clinician looks? Grad-CAM (which time points drove the
decision), channel ablation (which channel it needs), and a model **trained without** the
chin EMG. The Grad-CAM helper is given.

In [ ]:
def grad_cam(model, x1):  # Grad-CAM importance over time for one mini (1,3,600)
    model.eval(); x1 = x1.to(device).requires_grad_(True)  # track input gradients
    logits, feat = model(x1, return_feat=True); feat.retain_grad()  # forward + keep feature grads
    model.zero_grad(); logits[0, 1].backward()  # backprop from the "any" score
    w = feat.grad.mean(dim=2, keepdim=True)  # importance of each feature map
    cam = F.relu((w * feat).sum(1))[0]; cam = cam / (cam.max() + 1e-8)  # weighted sum -> 0..1
    cam = F.interpolate(cam[None, None], size=600, mode="linear", align_corners=False)[0, 0]  # upsample to 600
    return cam.detach().cpu().numpy()  # importance curve

### 4.1 — Grad-CAM on a positive RBD mini
Grad-CAM gives **one importance curve** over the 3-second window (the network mixes the
three channels together in its first layer, so the importance is shared across them). Draw
it as a **continuous colour heat map** behind each channel's trace — brighter = that moment
mattered more to the "RBD" decision — instead of an on/off threshold. On an RBD mini the
heat should sit on the **chin-EMG bursts**.

> *Prompt:* "Let's see where the network looked. Pick one three-second clip from a test
> patient who truly has activity, and use the Grad-CAM helper above to get an importance
> value at each moment (between 0 and 1). Draw the three signals — brain wave, eye movement,
> chin muscle — stacked on top of one another, and behind each one paint a smooth colour
> 'heat map' of that importance over time, warmer meaning the network relied on that moment
> more (let the colour vary smoothly — no on/off cut-off). Lay the real signal as a thin
> dark line on top, and add one shared colour scale. We're hoping the warm zones sit right
> on the chin-muscle bursts."

In [ ]:
# 🧑‍💻 Your turn — generate this cell with Colab's Gemini using the prompt above.
# (Click "Generate" / the ✨ in the cell, paste the prompt, run it; if it errors,
#  paste the red error text back to Gemini and ask it to fix it.)

### 4.2 — Channel ablation
Zero one channel at a time on the test set and measure the accuracy drop.

> *Prompt:* "Test which signal the model actually relies on. Start from its accuracy on the
> test clips. Then, one signal at a time, blank that signal out (set it to zero) across all
> the test clips and re-measure the accuracy. Draw a bar chart of how much accuracy is lost
> for each of the three — brain wave, eye movement, chin muscle — and print the numbers. A
> big drop means the model leans heavily on that signal."

In [ ]:
# 🧑‍💻 Your turn — generate this cell with Colab's Gemini using the prompt above.
# (Click "Generate" / the ✨ in the cell, paste the prompt, run it; if it errors,
#  paste the red error text back to Gemini and ask it to fix it.)

### 4.3 — Train a model *without* the chin EMG
The toughest test: hide the chin muscle from the model during training too. The brain-wave
channel carries a little muscle contamination, so a flexible model *might* still recover a
bit. We give it a slightly **wider** version of the same network (more filters); a short run
(about 10 epochs) is enough to see the point. We also turn **augmentation off** here: this
model must hunt for faint EMG traces leaking into the EEG, and adding noise on top of an
already-zeroed EMG channel would only get in the way.

> *Prompt:* "A tougher test: hide the chin muscle from the model completely. Do just what the
> full-data training did earlier, but first make a copy of the recordings with the
> chin-muscle signal set to zero (call it `X_noemg`), then train on that copy — a fresh,
> slightly wider network, a short run of about 10 epochs, no augmentation — and keep the
> trained model as `model_noemg`. Draw its learning curves (error and accuracy over the
> epochs, training and validation), so we can see how it does on just the brain wave and the
> eye movement."

In [ ]:
# 🧑‍💻 Your turn — generate this cell with Colab's Gemini using the prompt above.
# (Click "Generate" / the ✨ in the cell, paste the prompt, run it; if it errors,
#  paste the red error text back to Gemini and ask it to fix it.)

Now (**given, just run it**) we score it on the test set — with the chin muscle hidden there
too — and print the full model's numbers next to it, to see how much the chin EMG was really
worth.

In [ ]:
report_test(model_noemg, "CNN (no chin EMG)", Xsrc=X_noemg)  # test CM + metrics vs the rule (scored WITHOUT the EMG)
print(f"for contrast, the full model (with chin EMG) scored:  "  # how much did the EMG matter?
      f"sensitivity {full_res['sens']:.0%} | specificity {full_res['spec']:.0%} | accuracy {full_res['acc']:.0%}")

### 4.4 — EEG-only: is 18 % still the right line?
Our 18 % cut-off was calibrated on the **chin muscle**. A model that never sees the chin
muscle may place patients and controls at completely different levels, so the same line
might no longer separate them. Look at the **test** subjects two ways, side by side: what the
EEG-only model *predicts* for each, and what each one's *true* labelled RSWA percentage is.
If the predicted dots for patients and controls overlap around 18 %, an EEG-only test would
need its own cut-off.

> *Prompt:* "For just the held-out test subjects, make two side-by-side dot plots (controls
> in one column, RBD in the other, dots spread out sideways, a dashed line at 18 %). On the
> left, plot what the EEG-only model *predicts* for each subject — the percentage of their
> REM minis it flags as having activity. On the right, plot each subject's *true* labelled
> percentage, for the same subjects. Then we can see whether 18 % still separates the two
> groups on the left, or whether EEG alone would want a different line."

In [ ]:
# 🧑‍💻 Your turn — generate this cell with Colab's Gemini using the prompt above.
# (Click "Generate" / the ✨ in the cell, paste the prompt, run it; if it errors,
#  paste the red error text back to Gemini and ask it to fix it.)

### Wrap-up
- A 1-D CNN learns RSWA from the raw signal; its per-mini output aggregates to the same
  "any" density (and diagnosis) as the clinical rule.
- Little data → **overfitting**; augmentation and more data close the gap.
- Grad-CAM lands on the EMG bursts, ablation shows the model needs the **chin EMG**, and
  a model trained without it degrades — it looks where a clinician looks.

**Thank you!** — and you built most of it by vibe-coding.

---
## References
1. American Academy of Sleep Medicine. *International Classification of Sleep Disorders*, 3rd ed. 2014.
2. Berry RB, et al.; AASM. *The AASM Manual for the Scoring of Sleep and Associated Events*, v2.1. 2012.
3. Lapierre O, Montplaisir J. Polysomnographic features of REM sleep behavior disorder. *Neurology*. 1992;42(7):1371–1374.
4. Ferri R, et al. A quantitative statistical analysis of the submentalis muscle EMG amplitude during sleep. *J Sleep Res*. 2008;17(1):89–100.
5. Ferri R, et al. Improved computation of the atonia index in normal controls and patients with RBD. *Sleep Med*. 2010;11(9):947–949.
6. Frauscher B, et al.; SINBAR Group. Normative EMG values during REM sleep for the diagnosis of RBD. *Sleep*. 2012;35(6):835–847.
7. McCarter SJ, et al. Diagnostic thresholds for quantitative REM sleep phasic burst duration, phasic and tonic muscle activity, and REM atonia index. *Sleep*. 2014;37(10):1649–1662.
8. Leclair-Visonneau L, et al. Contemporary diagnostic visual and automated polysomnographic REM sleep without atonia thresholds in isolated RBD. *J Clin Sleep Med*. 2024;20(2):279–291. doi:10.5664/jcsm.10862.
9. Schenck CH, et al. Delayed emergence of a parkinsonian disorder or dementia in 81% of older men initially diagnosed with idiopathic RBD: a 16-year update. *Sleep Med*. 2013;14(8):744–748.
10. Iranzo A, et al. Neurodegenerative disease status and post-mortem pathology in idiopathic RBD. *Lancet Neurol*. 2013;12(5):443–453.